# Debugging Training

Notebook for **Chapter 4** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch04.html

Tip: change a number, predict the output, then run the cell again.

## 4.1 Sanity Tests: Initial Loss and Overfit One Batch

In [ ]:
m = MLP([2,16,2], seed=0)
print("untrained loss", round(m.loss(X,y),4), "ln2", round(np.log(2),4))
rng = np.random.default_rng(0)
Xr = rng.normal(size=(5000,20)); yr = rng.integers(0,10,5000)
m10 = MLP([20,64,10], seed=0)
print("10-class untrained", round(m10.loss(Xr,yr),4), "ln10", round(np.log(10),4))
m10.W[-1] *= 0.01
print("last layer x0.01", round(m10.loss(Xr,yr),4))

untrained loss 0.6009 ln2 0.6931
10-class untrained 3.2477 ln10 2.3026
last layer x0.01 2.3028


In [ ]:
xb = rng.normal(size=(16,20)); yb = rng.integers(0,4,16)
def train(lr, steps=300):
    m = MLP([20,64,4], seed=0); out = []
    for t in range(steps+1):
        m.forward(xb); gW, gb = m.backward(yb)
        for k in range(len(m.W)):
            m.W[k] -= lr*gW[k]; m.b[k] -= lr*gb[k]
        if t in (0,50,100,200,300): out.append(round(float(m.loss(xb,yb)),4))
    return out
print("lr 0.1 ", train(0.1)); print("lr 0.001", train(0.001)); print("lr 5.0 ", train(5.0))

lr 0.1  [1.7889, 0.0663, 0.0273, 0.0114, 0.0069]
lr 0.001 [2.2439, 2.0127, 1.8082, 1.4715, 1.2028]
lr 5.0  [4.1827, 0.4986, 0.4986, 0.4986, 0.4986]


## 4.2 Vanishing and Exploding Gradients, Clipping and Dead ReLUs

In [ ]:
def gradnorms(act, dact, scale, L=10, w=64, seed=0):
    r = np.random.default_rng(seed); Xb = r.normal(size=(64,w))
    Ws = [r.normal(size=(w,w))*scale(w) for _ in range(L)]
    A = [Xb]; Z = []
    for W in Ws:
        Z.append(A[-1]@W); A.append(act(Z[-1]))
    d = np.ones_like(A[-1])/64; norms = []
    for k in reversed(range(L)):
        d = d*dact(Z[k]); norms.append(np.linalg.norm(A[k].T@d)); d = d@Ws[k].T
    return np.array(norms[::-1])
sig = lambda z: 1/(1+np.exp(-z))
print(gradnorms(sig, lambda z: sig(z)*(1-sig(z)), lambda n: np.sqrt(1/n))[[0,2,4,6,9]])
print(gradnorms(lambda z: np.maximum(0,z), lambda z: (z>0)*1.0, lambda n: np.sqrt(2/n))[[0,2,4,6,9]])

[0.     0.0004 0.0068 0.1216 7.6198]
[12.2857 29.9247 36.9966 41.4117 32.2434]


In [ ]:
r = np.random.default_rng(0); xx = r.normal(size=(1000,10)); W = r.normal(size=(10,50))*np.sqrt(2/10)
for bias in [0,-10]:
    z = xx@W + bias
    print("bias", bias, "zero fraction:", round(float((z<=0).mean()),3),
          "never fire:", int(((z>0).sum(0)==0).sum()))

bias 0 zero fraction: 0.501 never fire: 0
bias -10 zero fraction: 1.0 never fire: 50


## 4.3 Loss Curves, Spikes and the Debugging Checklist

## 4.4 The Training Step as a Machine: Eight Places It Breaks

In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons()
def train(bug=None, lr=0.5, steps=200):
    m = MLP([2, 16, 2], seed=0); rng = np.random.default_rng(1)
    Xt = X * 1000 if bug == "scale" else X
    for t in range(steps):
        yt = rng.permutation(y) if bug == "shuffle" else y
        m.forward(Xt); gW, gb = m.backward(yt)
        s = -1 if bug == "sign" else 1
        for k in range(len(m.W)):
            m.W[k] -= s * lr * gW[k]
            if bug != "nobias": m.b[k] -= s * lr * gb[k]
    return m.loss(Xt, y), m.acc(Xt, y)
with np.errstate(all="ignore"):
    for bug in [None, "shuffle", "scale", "sign", "nobias"]:
        loss, acc = train(bug)
        print(f"{str(bug):8s} loss {loss:8.4f}  accuracy {acc:.3f}")

None     loss   0.1031  accuracy 0.970
shuffle  loss   0.7011  accuracy 0.435
scale    loss      nan  accuracy 0.500
sign     loss      inf  accuracy 0.395
nobias   loss   0.2351  accuracy 0.875


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons()
def run(zero_grad, lr, steps=200):
    m = MLP([2, 16, 2], seed=0); acc = None; used = []
    for t in range(steps):
        m.forward(X); gW, gb = m.backward(y)
        if zero_grad or acc is None:
            acc = ([g.copy() for g in gW], [g.copy() for g in gb])
        else:
            for k in range(len(gW)): acc[0][k] += gW[k]; acc[1][k] += gb[k]
        used.append(gnorm(*acc))
        for k in range(len(m.W)): m.W[k] -= lr * acc[0][k]; m.b[k] -= lr * acc[1][k]
    return used, m.loss(X, y)
with np.errstate(all="ignore"):
    for lr in (0.5, 2.0):
        for zg in (True, False):
            used, loss = run(zg, lr)
            print(f"lr {lr} zero_grad {zg!s:5}  norm used at steps 1,10,50,200: "
                  f"{used[0]:.3f} {used[9]:.3f} {used[49]:.3f} {used[199]:.3f}   final loss {loss + 0.0:.4f}")

lr 0.5 zero_grad True   norm used at steps 1,10,50,200: 0.658 0.096 0.037 0.043   final loss 0.1031
lr 0.5 zero_grad False  norm used at steps 1,10,50,200: 0.658 1.208 1.532 1.533   final loss 0.0000
lr 2.0 zero_grad True   norm used at steps 1,10,50,200: 0.658 0.181 0.243 0.005   final loss 0.0080
lr 2.0 zero_grad False  norm used at steps 1,10,50,200: 0.658 1.093 1.052 1.191   final loss 0.3730


In [ ]:
import numpy as np
def audit(Xtr, ytr, Xva, yva, n_classes):
    print("shapes      :", Xtr.shape, ytr.shape, Xva.shape, yva.shape)
    print("dtype       :", Xtr.dtype, ytr.dtype)
    print("X range     : min %.2f max %.2f mean %.2f std %.2f" % (np.nanmin(Xtr), np.nanmax(Xtr), np.nanmean(Xtr), np.nanstd(Xtr)))
    print("nan / inf   :", int(np.isnan(Xtr).sum()), int(np.isinf(Xtr).sum()))
    print("label range :", ytr.min(), "to", ytr.max(), "(expected 0 to %d)" % (n_classes - 1))
    print("class counts:", np.bincount(ytr, minlength=n_classes).tolist())
    tr = {row.tobytes() for row in np.nan_to_num(Xtr)}
    dup = sum(row.tobytes() in tr for row in np.nan_to_num(Xva))
    print("val rows that also appear in train:", dup, "of", len(Xva))
rng = np.random.default_rng(0)
Xtr = rng.normal(size=(200, 3)); ytr = (Xtr[:, 0] > 0).astype(int); ytr[:5] = 3
Xtr[7, 1] = np.nan; Xtr[:, 2] *= 500
Xva = np.vstack([rng.normal(size=(30, 3)), Xtr[10:30]]); yva = (Xva[:, 0] > 0).astype(int)
audit(Xtr, ytr, Xva, yva, n_classes=2)

shapes      : (200, 3) (200,) (50, 3) (50,)
dtype       : float64 int64
X range     : min -1483.59 max 999.75 mean -7.50 std 281.83
nan / inf   : 1 0
label range : 0 to 3 (expected 0 to 1)
class counts: [104, 91, 0, 5]
val rows that also appear in train: 20 of 50


## 4.5 Instruments: Gradient Norms, Update Ratios, Activation Statistics and Gradient Checks

In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons()
m = MLP([2, 16, 16, 2], seed=0); m.forward(X); gW, gb = m.backward(y)
print("weight norms :", [round(float(np.linalg.norm(W)), 3) for W in m.W])
print("grad norms   :", [round(float(np.linalg.norm(g)), 4) for g in gW])
for lr in (0.001, 0.01, 0.1, 0.5, 5.0):
    ratios = [lr * np.linalg.norm(g) / np.linalg.norm(W) for g, W in zip(gW, m.W)]
    print(f"lr {lr:<6}", " ".join(f"{r:.5f}" for r in ratios))

weight norms : [4.599, 5.926, 2.008]
grad norms   : [0.4741, 1.5329, 0.7645]
lr 0.001  0.00010 0.00026 0.00038
lr 0.01   0.00103 0.00259 0.00381
lr 0.1    0.01031 0.02587 0.03808
lr 0.5    0.05155 0.12933 0.19038
lr 5.0    0.51545 1.29328 1.90385


In [ ]:
import numpy as np
rng = np.random.default_rng(0); X0 = rng.normal(size=(512, 64))
print("init   layer  mean   std  dead_fraction")
for name, std in [("He", np.sqrt(2/64)), ("small", 0.05), ("big", 0.4)]:
    r = np.random.default_rng(1); a = X0
    for layer in range(1, 9):
        a = np.maximum(0, a @ r.normal(0, std, (64, 64)))
        if layer in (1, 4, 8):
            print(f"{name:6s} {layer:5d} {a.mean():7.3f} {a.std():7.3f} {(a == 0).mean():8.3f}")

init   layer  mean   std  dead_fraction
He         1   0.563   0.830    0.501
He         4   0.474   0.758    0.519
He         8   0.366   0.701    0.602
small      1   0.159   0.235    0.501
small      4   0.003   0.005    0.519
small      8   0.000   0.000    0.602
big        1   1.275   1.877    0.501
big        4  12.429  19.879    0.519
big        8 251.590 481.981    0.602


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

class BuggyMLP(MLP):
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = d @ s.W[k].T          # BUG: ReLU slope missing
        return gW, gb
def grad_check(m, X, y, tries=6, eps=1e-5, seed=0):
    r = np.random.default_rng(seed); m.forward(X); gW, _ = m.backward(y); worst = 0
    for _ in range(tries):
        k = r.integers(len(m.W)); i = r.integers(m.W[k].shape[0]); j = r.integers(m.W[k].shape[1])
        m.W[k][i, j] += eps; lp = m.loss(X, y); m.W[k][i, j] -= 2*eps; lm = m.loss(X, y); m.W[k][i, j] += eps
        num = (lp - lm) / (2*eps); ana = gW[k][i, j]
        worst = max(worst, abs(num - ana) / max(abs(num) + abs(ana), 1e-12))
    return worst
X, y = moons()
for cls in (MLP, BuggyMLP):
    print(cls.__name__, "worst relative error: %.2e" % grad_check(cls([2, 16, 16, 2], seed=0), X[:20], y[:20]))

MLP worst relative error: 2.02e-10
BuggyMLP worst relative error: 1.00e+00


## 4.6 When the Loss Is nan or inf: Numerical Stability

In [ ]:
import numpy as np
z = np.array([1000.0, 0.0, -1000.0])
with np.errstate(all="ignore"):
    naive = np.exp(z) / np.exp(z).sum()
print("naive softmax :", naive)
shift = np.exp(z - z.max()); print("shifted       :", shift / shift.sum())
lse = z.max() + np.log(np.exp(z - z.max()).sum())
print("logsumexp     :", lse, " log-probs:", z - lse)
with np.errstate(all="ignore"):
    print("exp(709), exp(710):", np.exp(709.0), np.exp(710.0))
    print("log(0), log(1e-320):", np.log(0.0), np.log(1e-320))

naive softmax : [nan  0.  0.]
shifted       : [1. 0. 0.]
logsumexp     : 1000.0  log-probs: [    0. -1000. -2000.]
exp(709), exp(710): 8.218407461554972e+307 inf
log(0), log(1e-320): -inf -736.8272408909739


In [ ]:
import numpy as np
def bce_naive(logit, y):
    p = 1 / (1 + np.exp(-logit))
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))
def bce_stable(logit, y):
    return np.maximum(logit, 0) - logit * y + np.log1p(np.exp(-np.abs(logit)))
with np.errstate(all="ignore"):
    for logit, y in [(2.0, 1), (-40.0, 1), (40.0, 0), (-800.0, 1), (800.0, 0)]:
        print(f"logit {logit:7.1f} label {y}  naive {bce_naive(logit, y):8.4f}  stable {bce_stable(logit, y):8.4f}")

logit     2.0 label 1  naive   0.1269  stable   0.1269
logit   -40.0 label 1  naive  40.0000  stable  40.0000
logit    40.0 label 0  naive      inf  stable  40.0000
logit  -800.0 label 1  naive      inf  stable 800.0000
logit   800.0 label 0  naive      inf  stable 800.0000


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons(); Xs = X * 1000
m = MLP([2, 16, 2], seed=0); lr = 0.5; first_nan = None
with np.errstate(all="ignore"):
    for t in range(1, 101):
        m.forward(Xs); gW, gb = m.backward(y)
        loss, g = float(m.loss(Xs, y)), gnorm(gW, gb)
        if t <= 8: print(f"step {t:3d} loss {loss:12.4f} grad_norm {g:.3e}")
        if np.isnan(loss) or np.isnan(g):
            first_nan = t; break
        for k in range(2): m.W[k] -= lr * gW[k]; m.b[k] -= lr * gb[k]
print("first nan at step", first_nan)
print("weights finite:", all(np.isfinite(W).all() for W in m.W), " largest |weight|: %.2e" % max(np.abs(W).max() for W in m.W))

step   1 loss      20.7672 grad_norm 5.089e+02
step   2 loss          inf grad_norm 1.356e+05
step   3 loss          inf grad_norm 2.062e+07
step   4 loss          inf grad_norm 8.108e+09
step   5 loss          inf grad_norm 1.272e+12
step   6 loss          inf grad_norm 5.005e+14
step   7 loss          inf grad_norm 7.851e+16
step   8 loss          inf grad_norm 3.090e+19
first nan at step 67
weights finite: True  largest |weight|: 1.95e+155


In [ ]:
import numpy as np
with np.errstate(all="ignore"):
    big = np.float16(60000.0) * np.float16(2.0)
    print("60000 * 2 in float16     :", big, "  inf - inf:", big - big)
    print("1e-8 in float16          :", np.float16(1e-8))
    print("2048 + 1 in float16      :", np.float16(2048) + np.float16(1))
    print("2048 + 1 in float32      :", np.float32(2048) + np.float32(1))
    g = 3e-8
    for scale in (1, 1024, 65536):
        s = np.float16(g * scale)
        print(f"grad {g} scaled by {scale:6d} -> float16 {s!s:10}  unscaled {float(s)/scale:.3e}")

60000 * 2 in float16     : inf   inf - inf: nan
1e-8 in float16          : 0.0
2048 + 1 in float16      : 2.048e+03
2048 + 1 in float32      : 2049.0
grad 3e-08 scaled by      1 -> float16 6e-08       unscaled 5.960e-08
grad 3e-08 scaled by   1024 -> float16 3.07e-05    unscaled 2.998e-08
grad 3e-08 scaled by  65536 -> float16 0.001966    unscaled 3.001e-08


In [ ]:
import numpy as np
def softmax(z): e = np.exp(z - z.max()); return e / e.sum()
def ce_from_logits(z, y): return -np.log(softmax(z)[y])
z = np.array([4.0, 0.0, -2.0]); y = 0
print("correct loss (logits)        :", round(ce_from_logits(z, y), 4))
p = softmax(z)
print("probabilities                :", np.round(p, 4))
print("loss if probs are fed as logits:", round(ce_from_logits(p, y), 4))
g_right = softmax(z) - np.eye(3)[y]
g_wrong = softmax(p) - np.eye(3)[y]
print("gradient size right / wrong  :", round(np.linalg.norm(g_right), 4), round(np.linalg.norm(g_wrong), 4))

correct loss (logits)        : 0.0206
probabilities                : [0.9796 0.0179 0.0024]
loss if probs are fed as logits: 0.5645
gradient size right / wrong  : 0.0273 0.5283


## 4.7 The Learning Rate in Practice: Range Test, Spikes, Clipping and Batch Size

In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons(); m = MLP([2, 16, 2], seed=0)
lrs = np.logspace(-4, 2, 120); L = []
with np.errstate(all="ignore"):
    for lr in lrs:
        m.forward(X); gW, gb = m.backward(y); L.append(m.loss(X, y))
        for k in range(2): m.W[k] -= lr * gW[k]; m.b[k] -= lr * gb[k]
L = np.array(L); j = int(L.argmin())
for i in range(0, 120, 12): print(f"lr {lrs[i]:9.5f}  loss {L[i]:.4f}")
print("minimum loss %.4f at lr %.3f" % (L[j], lrs[j]))
print("first lr where loss > 1:", round(float(lrs[np.argmax(L > 1)]), 3))
print("suggested range: %.3f to %.3f" % (lrs[j] / 10, lrs[j] / 3))

lr   0.00010  loss 0.6009
lr   0.00040  loss 0.5999
lr   0.00162  loss 0.5957
lr   0.00653  loss 0.5796
lr   0.02631  loss 0.5264
lr   0.10598  loss 0.4128
lr   0.42683  loss 0.3008
lr   1.71907  loss 0.2501
lr   6.92367  loss 0.7645
lr  27.88548  loss 8.4956
minimum loss 0.2327 at lr 3.450
first lr where loss > 1: 13.895
suggested range: 0.345 to 1.150


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons()
def run(clip=None, steps=300, lr=0.3):
    m = MLP([2, 16, 2], seed=0); r = np.random.default_rng(0); L = []; norms = []; fired = 0
    for t in range(steps):
        idx = r.integers(0, len(X), 32); xb = X[idx].copy(); yb = y[idx]
        if t in (100, 200): xb = xb * 30; yb = 1 - yb          # two corrupted batches
        m.forward(xb); gW, gb = m.backward(yb); n = gnorm(gW, gb); norms.append(n)
        if clip and n > clip:
            f = clip / n; gW = [g * f for g in gW]; gb = [g * f for g in gb]; fired += 1
        for k in range(2): m.W[k] -= lr * gW[k]; m.b[k] -= lr * gb[k]
        L.append(float(m.loss(X, y)))
    return L, norms, fired
with np.errstate(all="ignore"):
    for clip in (None, 1.0):
        L, norms, fired = run(clip)
        print(f"clip {clip}: loss at 99/100/101/200/201/299 = "
              + " ".join(f"{L[i]:.3f}" for i in (99, 100, 101, 200, 201, 299))
              + f" | clip fired {fired}/300 | max norm {max(norms):.1f} | median norm {np.median(norms):.3f}")

clip None: loss at 99/100/101/200/201/299 = 0.227 1.071 1.418 170.420 inf 0.606 | clip fired 0/300 | max norm 289.2 | median norm 0.323
clip 1.0: loss at 99/100/101/200/201/299 = 0.227 0.247 0.249 0.207 0.186 0.134 | clip fired 2/300 | max norm 72.5 | median norm 0.172


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons()
def run(clip, steps=300, lr=0.3):
    m = MLP([2, 16, 2], seed=0); r = np.random.default_rng(0); fired = 0; steplen = []
    for t in range(steps):
        idx = r.integers(0, len(X), 32); m.forward(X[idx]); gW, gb = m.backward(y[idx]); n = gnorm(gW, gb)
        f = min(1.0, clip / n); fired += n > clip; steplen.append(lr * n * f)
        for k in range(2): m.W[k] -= lr * f * gW[k]; m.b[k] -= lr * f * gb[k]
    return fired / steps, np.mean(steplen), m.loss(X, y)
for clip in (1.0, 0.3, 0.1, 0.03):
    frac, sl, loss = run(clip)
    print(f"clip {clip:5}: fired on {frac:5.1%} of steps, mean step length {sl:.4f}, final loss {loss:.4f}")

clip   1.0: fired on  0.0% of steps, mean step length 0.0615, final loss 0.1220
clip   0.3: fired on 18.7% of steps, mean step length 0.0552, final loss 0.1289
clip   0.1: fired on 85.3% of steps, mean step length 0.0291, final loss 0.1892
clip  0.03: fired on 100.0% of steps, mean step length 0.0090, final loss 0.2503


## 4.8 Reading the Curves: Overfitting, Underfitting, Leakage and Misleading Metrics

In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

def fit(sizes, ntr, noise, lr, steps, wd=0.0, seed=0):
    Xa, ya = moons(ntr // 2, noise, seed); Xv, yv = moons(300, noise, seed + 7)
    m = MLP(sizes, seed=0); va = []
    for t in range(steps):
        m.forward(Xa); gW, gb = m.backward(ya)
        for k in range(len(m.W)): m.W[k] -= lr * (gW[k] + wd * m.W[k]); m.b[k] -= lr * gb[k]
        va.append(m.loss(Xv, yv))
    return m.loss(Xa, ya), np.array(va), m.acc(Xv, yv)
cases = {"A healthy": ([2,16,2], 200, .2, .3, 300), "B overfit": ([2,128,128,2], 24, .35, .03, 800),
         "C linear": ([2,2], 200, .1, .5, 300), "D lr too high": ([2,16,2], 200, .1, 12., 300)}
with np.errstate(all="ignore"):
    for name, (sz, n, nz, lr, st) in cases.items():
        tr, va, acc = fit(sz, n, nz, lr, st)
        print(f"{name:14s} train {tr:.3f}  val {va[-1]:.3f}  best val {va.min():.3f} at step {va.argmin():3d}  val acc {acc:.3f}")

A healthy      train 0.176  val 0.195  best val 0.195 at step 299  val acc 0.935
B overfit      train 0.207  val 0.572  best val 0.424 at step  42  val acc 0.752
C linear       train 0.266  val 0.272  best val 0.272 at step 299  val acc 0.868
D lr too high  train 0.189  val 0.246  best val 0.049 at step 175  val acc 0.917


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
y = (rng.random(2000) < 0.05).astype(int)               # 5% positives
pred_all_zero = np.zeros_like(y)
def report(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum())
    rec = tp / max(tp + fn, 1); prec = tp / max(tp + fp, 1)
    return f"accuracy {(y == p).mean():.3f}  recall {rec:.3f}  precision {prec:.3f}"
print("positives in data:", int(y.sum()), "of", len(y))
print("always predict 0 :", report(y, pred_all_zero))
noisy = np.where(rng.random(2000) < 0.85, y, 1 - y)     # a real but imperfect model: 85% right on each row
print("85% model        :", report(y, noisy))

positives in data: 110 of 2000
always predict 0 : accuracy 0.945  recall 0.000  precision 0.000
85% model        : accuracy 0.860  recall 0.891  precision 0.268


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
rng = np.random.default_rng(0)
X = rng.normal(size=(400, 5)); y = rng.integers(0, 2, 400)       # labels are pure noise
Xtr, ytr, Xva, yva = X[:300], y[:300], X[300:], y[300:]
knn = KNeighborsClassifier(1).fit(Xtr, ytr)
print("clean split   : val accuracy %.3f (coin flip is 0.5)" % knn.score(Xva, yva))
Xva2 = np.vstack([Xva[:50], Xtr[:50]]); yva2 = np.r_[yva[:50], ytr[:50]]     # half the val rows leaked from train
print("50%% leaked    : val accuracy %.3f" % knn.score(Xva2, yva2))

clean split   : val accuracy 0.480 (coin flip is 0.5)
50% leaked    : val accuracy 0.730


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

def fit(wd, steps=800, patience=None):
    Xa, ya = moons(12, 0.35, 0); Xv, yv = moons(300, 0.35, 7)
    m = MLP([2, 128, 128, 2], seed=0); best = (9, 0); va = []
    for t in range(steps):
        m.forward(Xa); gW, gb = m.backward(ya)
        for k in range(3): m.W[k] -= 0.03 * (gW[k] + wd * m.W[k]); m.b[k] -= 0.03 * gb[k]
        va.append(m.loss(Xv, yv))
    return va
for wd in (0.0, 0.01, 0.1):
    va = fit(wd); print(f"weight decay {wd:<5}: final val {va[-1]:.3f}  best val {min(va):.3f} at step {int(np.argmin(va))}")

weight decay 0.0  : final val 0.572  best val 0.424 at step 42
weight decay 0.01 : final val 0.489  best val 0.424 at step 42
weight decay 0.1  : final val 0.518  best val 0.432 at step 33


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

Xa, ya = moons(12, 0.35, 0); Xv, yv = moons(300, 0.35, 7)
def train_es(patience, max_steps=800, lr=0.03):
    m = MLP([2, 128, 128, 2], seed=0); best = (np.inf, 0, None); wait = 0
    for t in range(max_steps):
        m.forward(Xa); gW, gb = m.backward(ya)
        for k in range(3): m.W[k] -= lr * gW[k]; m.b[k] -= lr * gb[k]
        v = m.loss(Xv, yv)
        if v < best[0] - 1e-4:
            best = (v, t, ([w.copy() for w in m.W], [b.copy() for b in m.b])); wait = 0
        else:
            wait += 1
            if wait >= patience: break
    m.W, m.b = best[2]
    return t, best[1], m.loss(Xv, yv), m.acc(Xv, yv)
for p in (5, 20, 100):
    stop, bst, v, a = train_es(p)
    print(f"patience {p:3d}: stopped at step {stop:3d}, restored step {bst:3d}, val loss {v:.3f}, val accuracy {a:.3f}")

patience   5: stopped at step  47, restored step  42, val loss 0.424, val accuracy 0.793
patience  20: stopped at step  62, restored step  42, val loss 0.424, val accuracy 0.793
patience 100: stopped at step 142, restored step  42, val loss 0.424, val accuracy 0.793


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons(150, 0.1, 0)
def fit(h, seed, steps=1500, lr=0.3):
    m = MLP([2, h, 2], seed=seed)
    for t in range(steps):
        m.forward(X); gW, gb = m.backward(y)
        for k in range(2): m.W[k] -= lr * gW[k]; m.b[k] -= lr * gb[k]
    m.forward(X); alive = int((m.a[1] > 0).any(0).sum())
    return m, alive
for h in (1, 2, 4, 16):
    m, alive = fit(h, 0)
    params = sum(w.size for w in m.W) + sum(b.size for b in m.b)
    print(f"hidden width {h:2d}: params {params:3d}  loss {m.loss(X, y):.3f}  accuracy {m.acc(X, y):.3f}  units alive {alive}/{h}")
print("width 1 with six different seeds:")
for seed in range(6):
    m, alive = fit(1, seed)
    print(f"  seed {seed}: loss {m.loss(X, y):.3f}  accuracy {m.acc(X, y):.3f}  unit alive: {bool(alive)}")

hidden width  1: params   7  loss 0.693  accuracy 0.500  units alive 0/1
hidden width  2: params  12  loss 0.272  accuracy 0.873  units alive 2/2
hidden width  4: params  22  loss 0.226  accuracy 0.893  units alive 4/4
hidden width 16: params  82  loss 0.007  accuracy 1.000  units alive 16/16
width 1 with six different seeds:
  seed 0: loss 0.693  accuracy 0.500  unit alive: False
  seed 1: loss 0.272  accuracy 0.873  unit alive: True
  seed 2: loss 0.693  accuracy 0.500  unit alive: False
  seed 3: loss 0.272  accuracy 0.873  unit alive: True
  seed 4: loss 0.272  accuracy 0.873  unit alive: True
  seed 5: loss 0.272  accuracy 0.873  unit alive: True


## 4.9 How to Solve Any Debugging Training Problem

In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

rng = np.random.default_rng(0); Xr = rng.normal(size=(5000, 20)); yr = rng.integers(0, 10, 5000)
print("ln 10 =", round(np.log(10), 4))
for scale in (0.01, 0.1, 0.5, 1.0, 3.0):
    m = MLP([20, 64, 10], seed=0); m.W[-1] *= scale
    m.forward(Xr)
    print(f"last layer x{scale:<5}: initial loss {m.loss(Xr, yr):7.4f}")

ln 10 = 2.3026
last layer x0.01 : initial loss  2.3028
last layer x0.1  : initial loss  2.3149
last layer x0.5  : initial loss  2.5726
last layer x1.0  : initial loss  3.2477
last layer x3.0  : initial loss  7.3003


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

Xv, yv = moons(300, 0.35, 7)
for per_class in (6, 12, 24, 48, 96, 192):
    Xa, ya = moons(per_class, 0.35, 0); m = MLP([2, 128, 128, 2], seed=0); best = 9
    for t in range(800):
        m.forward(Xa); gW, gb = m.backward(ya)
        for k in range(3): m.W[k] -= 0.03 * gW[k]; m.b[k] -= 0.03 * gb[k]
    tr, va = m.loss(Xa, ya), m.loss(Xv, yv)
    print(f"{2*per_class:4d} training points: train {tr:.3f}  val {va:.3f}  gap {va - tr:.3f}  val acc {m.acc(Xv, yv):.3f}")

  12 training points: train 0.130  val 0.952  gap 0.822  val acc 0.638
  24 training points: train 0.207  val 0.572  gap 0.364  val acc 0.752
  48 training points: train 0.220  val 0.578  gap 0.359  val acc 0.762
  96 training points: train 0.221  val 0.383  gap 0.162  val acc 0.833
 192 training points: train 0.275  val 0.358  gap 0.083  val acc 0.853
 384 training points: train 0.286  val 0.360  gap 0.075  val acc 0.843


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons(150, 0.1, 0)
def poly(X): return np.c_[X, X**2, X[:, :1] * X[:, 1:]]
def fit(Xin, sizes, steps=3000, lr=0.3):
    m = MLP(sizes, seed=0)
    for t in range(steps):
        m.forward(Xin); gW, gb = m.backward(y)
        for k in range(len(m.W)): m.W[k] -= lr * gW[k]; m.b[k] -= lr * gb[k]
    return m.loss(Xin, y), m.acc(Xin, y)
for name, Xin, sizes in [("linear on raw x, y", X, [2, 2]), ("linear on x, y, x^2, y^2, xy", poly(X), [5, 2]), ("MLP 2-16-2 on raw x, y", X, [2, 16, 2])]:
    l, a = fit(Xin, sizes); print(f"{name:30s} loss {l:.3f}  accuracy {a:.3f}")

linear on raw x, y             loss 0.272  accuracy 0.873
linear on x, y, x^2, y^2, xy   loss 0.271  accuracy 0.870
MLP 2-16-2 on raw x, y         loss 0.003  accuracy 1.000


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

X, y = moons()
def train(inplace, steps=100, lr=0.5):
    m = MLP([2, 16, 2], seed=0)
    for t in range(steps):
        m.forward(X); gW, gb = m.backward(y)
        for W, g in zip(m.W, gW):
            if inplace: W -= lr * g
            else: W = W - lr * g              # rebinds the local name W; the model never changes
    return m.loss(X, y)
print("loss before training:", round(float(MLP([2,16,2], seed=0).loss(X, y)), 4))
print("buggy update (W = W - ...):", round(float(train(False)), 4))
print("in-place update (W -= ...):", round(float(train(True)), 4))

loss before training: 0.6009
buggy update (W = W - ...): 0.6009
in-place update (W -= ...): 0.2403


In [ ]:
import numpy as np
rng = np.random.default_rng(0); X = rng.normal(size=(1000, 10)); W = rng.normal(size=(10, 50)) * np.sqrt(2/10); b = -10.0
z = X @ W + b; up = np.ones_like(z) / 1000                       # pretend upstream gradient
def grad_W(slope): return X.T @ (up * slope(z))
relu = lambda z: (z > 0) * 1.0
leaky = lambda z: np.where(z > 0, 1.0, 0.01)
print("ReLU       gradient norm:", np.linalg.norm(grad_W(relu)))
print("Leaky 0.01 gradient norm:", round(float(np.linalg.norm(grad_W(leaky))), 4))

ReLU       gradient norm: 0.0
Leaky 0.01 gradient norm: 0.0075


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

Xa, ya = moons(100, 0.2, 0); Xv, yv = moons(300, 0.2, 7); out = []
for seed in range(8):
    m = MLP([2, 16, 2], seed=seed)
    for t in range(150):
        m.forward(Xa); gW, gb = m.backward(ya)
        for k in range(2): m.W[k] -= 0.3 * gW[k]; m.b[k] -= 0.3 * gb[k]
    out.append(m.loss(Xv, yv))
out = np.array(out); print("val loss per seed:", np.round(out, 3))
print("mean %.3f  std %.3f  range %.3f" % (out.mean(), out.std(ddof=1), out.max() - out.min()))

val loss per seed: [0.265 0.276 0.279 0.291 0.232 0.259 0.254 0.289]
mean 0.268  std 0.020  range 0.059


In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
Xa, ya = moons(100, 0.2, 0); Xv, yv = moons(300, 0.2, 7)
print("logistic regression baseline:", round(LogisticRegression().fit(Xa, ya).score(Xv, yv), 3))
print("sklearn MLP reference       :", round(MLPClassifier((16,), max_iter=2000, random_state=0).fit(Xa, ya).score(Xv, yv), 3))
m = MLP([2, 16, 2], seed=0)
for t in range(150):
    m.forward(Xa); gW, gb = m.backward(ya)
    for k in range(2): m.W[k] -= 0.3 * gW[k]; m.b[k] -= 0.3 * gb[k]
print("our NumPy MLP               :", round(m.acc(Xv, yv), 3))

logistic regression baseline: 0.848
sklearn MLP reference       : 0.852
our NumPy MLP               : 0.887


## 4.10 The Same Checks in PyTorch (not run here)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

def train_epoch(model, loader, optimizer, device, clip=1.0, log_every=50):
    model.train()                                   # 1
    clipped, steps = 0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)       # 2
        optimizer.zero_grad(set_to_none=True)       # 3
        logits = model(xb)                          # 4
        loss = F.cross_entropy(logits, yb)          # 5
        if not torch.isfinite(loss):                # 6
            raise RuntimeError(f"non-finite loss {loss.item()} at step {step}")
        loss.backward()                             # 7
        norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)   # 8
        clipped += int(norm > clip); steps += 1
        optimizer.step()                            # 9
        if step % log_every == 0:
            print(f"step {step:5d} loss {loss.item():.4f} grad_norm {norm.item():.3f}")
    print(f"clip fired on {clipped}/{steps} steps")

In [ ]:
import math, torch, torch.nn.functional as F

def sanity(model, loader, num_classes, device, steps=300, lr=1e-3):
    xb, yb = next(iter(loader)); xb, yb = xb[:16].to(device), yb[:16].to(device)
    print("shapes", tuple(xb.shape), tuple(yb.shape), "dtypes", xb.dtype, yb.dtype)
    print("x range", xb.min().item(), xb.max().item(), "labels", yb.min().item(), yb.max().item())
    model.train()
    with torch.no_grad():
        print("initial loss", F.cross_entropy(model(xb), yb).item(), "expected", math.log(num_classes))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    before = [p.detach().clone() for p in model.parameters()]
    for i in range(steps):
        opt.zero_grad(); loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step()
        if i == 0:
            moved = any(not torch.equal(a, p) for a, p in zip(before, model.parameters()))
            print("weights changed after one step:", moved)
    print("loss after", steps, "steps on 16 examples:", loss.item())

In [ ]:
import torch, torch.nn as nn

stats = {}
def make_hook(name):
    def hook(module, inputs, output):
        stats[name] = (output.mean().item(), output.std().item(), (output == 0).float().mean().item())
    return hook
handles = [m.register_forward_hook(make_hook(n)) for n, m in model.named_modules() if isinstance(m, nn.ReLU)]

model(xb)                                   # one forward pass fills `stats`
for name, (mean, std, zero) in stats.items():
    print(f"{name:12s} mean {mean:8.3f} std {std:8.3f} zeros {zero:5.2f}")

loss = loss_fn(model(xb), yb); loss.backward()
lr = 1e-3
for name, p in model.named_parameters():
    if p.grad is not None:
        print(f"{name:20s} grad {p.grad.norm().item():10.4e}  update ratio {lr * p.grad.norm().item() / p.norm().item():10.4e}")
for h in handles: h.remove()                # remove hooks when done

## 4.11 Going Deeper: Internals, Edge Cases and What Interviewers Probe

In [ ]:
import numpy as np

def moons(n=100, noise=0.1, seed=0):
    r = np.random.default_rng(seed)
    t = r.uniform(0, np.pi, n)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X = np.vstack([a, b]) + r.normal(0, noise, (2*n, 2))
    return X, np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1: s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True)); s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X); return -np.log(P[np.arange(len(y)), y]).mean()
    def acc(s, X, y):
        return (s.forward(X).argmax(1) == y).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0: d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

def gnorm(gW, gb):
    return float(np.sqrt(sum((g**2).sum() for g in gW) + sum((g**2).sum() for g in gb)))

rng = np.random.default_rng(0); K = 10
for sigma in (0.5, 1.0, 1.45, 2.0):
    z = rng.normal(0, sigma, size=(200000, K)); y = rng.integers(0, K, 200000)
    lse = np.log(np.exp(z - z.max(1, keepdims=True)).sum(1)) + z.max(1)
    loss = (lse - z[np.arange(len(y)), y]).mean()
    theory = np.log(K) + sigma**2 * (K - 1) / (2 * K)
    print(f"sigma {sigma:4}: simulated {loss:.4f}  formula {theory:.4f}")
loss0 = 3.2477
sigma2 = 2 * (loss0 - np.log(K)) * K / (K - 1)
print("from observed loss 3.2477: sigma^2 = %.3f, sigma = %.3f" % (sigma2, sigma2 ** 0.5))
r = np.random.default_rng(0); Xr = r.normal(size=(5000, 20))
m = MLP([20, 64, 10], seed=0); m.forward(Xr)
z = np.log(m.a[-1]); z -= z.mean(1, keepdims=True)
print("measured std of the network's centred logits: %.3f" % z.std())

sigma  0.5: simulated 2.4140  formula 2.4151
sigma  1.0: simulated 2.7274  formula 2.7526
sigma 1.45: simulated 3.1497  formula 3.2487
sigma  2.0: simulated 3.7711  formula 4.1026
from observed loss 3.2477: sigma^2 = 2.100, sigma = 1.449
measured std of the network's centred logits: 1.521


In [ ]:
import numpy as np
def adam_steps(grads, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
    m = v = 0.0; out = []
    for t, g in enumerate(grads, 1):
        m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g * g
        out.append(lr * (m / (1 - b1**t)) / (np.sqrt(v / (1 - b2**t)) + eps))
    return np.array(out)
for g in (1e-6, 1e-2, 1.0, 100.0):
    print(f"constant gradient {g:<8}: Adam step {adam_steps([g] * 3)[-1]:.2e}   SGD step (lr 1e-3): {1e-3 * g:.1e}")
grads = np.ones(6000); grads[5000] = 1000.0                   # steady gradient 1, one spike of 1000
st = adam_steps(grads)
print("Adam step before the spike  :", f"{st[4999]:.2e}")
print("Adam step at the spike      :", f"{st[5000]:.2e}   (= {st[5000] / 1e-3:.2f} x lr)")
for k in (1, 10, 100, 500, 999):
    print(f"Adam step {k:3d} steps later  :", f"{st[5000 + k]:.2e}")

constant gradient 1e-06   : Adam step 9.90e-04   SGD step (lr 1e-3): 1.0e-09
constant gradient 0.01    : Adam step 1.00e-03   SGD step (lr 1e-3): 1.0e-05
constant gradient 1.0     : Adam step 1.00e-03   SGD step (lr 1e-3): 1.0e-03
constant gradient 100.0   : Adam step 1.00e-03   SGD step (lr 1e-3): 1.0e-01
Adam step before the spike  : 1.00e-03
Adam step at the spike      : 3.18e-03   (= 3.18 x lr)
Adam step   1 steps later  : 2.87e-03
Adam step  10 steps later  : 1.13e-03
Adam step 100 steps later  : 3.32e-05
Adam step 500 steps later  : 4.05e-05
Adam step 999 steps later  : 5.20e-05


In [ ]:
import numpy as np
rng = np.random.default_rng(0); x = (rng.normal(size=100000) * 10 ** rng.uniform(-3, 3, 100000)).astype(np.float32)
a = np.float32(0)
for v in x: a += v
b = np.float32(0)
for v in x[::-1]: b += v
print("float32 sum forward :", a); print("float32 sum backward:", b)
print("float64 reference   :", x.astype(np.float64).sum()); print("numpy pairwise sum  :", x.sum())

float32 sum forward : -12342.822
float32 sum backward: -12342.848
float64 reference   : -12342.746260999818
numpy pairwise sum  : -12342.742


In [ ]:
import numpy as np
K, eps = 10, 0.1
target = np.full(K, eps / K); target[0] += 1 - eps
floor = -(target * np.log(target)).sum()
print("smoothed target:", target[:2], "...")
print("lowest possible loss with label smoothing 0.1 on 10 classes: %.4f" % floor)
print("with a perfect one-hot target the floor is 0.0")

smoothed target: [0.91 0.01] ...
lowest possible loss with label smoothing 0.1 on 10 classes: 0.5003
with a perfect one-hot target the floor is 0.0


## 4.12 Mixed-Practice Drill and Self-Test

## Chapter recap

## Practice set

**1.** (easy) What initial loss do you expect for a 5-class problem? **Answer:** `ln 5 = 1.609`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch04.html